# Task 1 — Organizational memory from the completed CLI runs

This is the main Task 1 results walkthrough. I generated the memory with the
command-line pipeline: **Luna judged the base episodes, facts and outcomes;
Astra synthesized only the patterns.** This notebook reads that completed
result from `artifacts/memory/memory.json` and demonstrates the five Task 1
capabilities: episodes, facts/outcomes, patterns, recall and retention.

**Run all cells without provider access.** The executable cells make no model
calls, rebuild no memory, and write no artifacts. They need the committed memory
and Python modules, not `KEP_2026.json`, credentials, provider caches or old runs.
Saved cell outputs make the result inspectable without executing the notebook.

[Methodology](METHODOLOGY.md#task-1-organizational-memory-methodology) explains
the design and trade-offs. [Data understanding](DATA_UNDERSTANDING.md) connects
source-data problems to safeguards and surviving errors. Examples here are
selected development inspections, not an independently labeled quality benchmark.

## 1. CLI workflow used to generate `memory.json`

The generation stages were `run_task1.py` followed by `run_abstraction.py`.
The saved metadata records the models, batch size, reasoning effort, packet cap,
cache locations and completed coverage. It does **not** retain the original shell
command lines. The following commands reconstruct the recorded configuration and
historical output paths; they are not a verbatim shell-history transcript.

**Stage A — Luna base memory.** Review the full graph with one candidate per
batch, then construct episodes, facts, outcome assessments and retention decisions.
The final saved invocation had a 300-call allowance. The run resumed across
invocations and records 838 cumulative requests; 300 was not the total build cost.

```bash
MEMORY_CACHE_DIR=artifacts/cache-production-batch1 \
MEMORY_CODEX_REASONING_EFFORT=medium \
python run_task1.py \
  --input KEP_2026.json \
  --mode hosted --provider codex --model gpt-5.6-luna \
  --batch-size 1 --max-calls 300 --max-packet-chars 100000 \
  --output artifacts/hosted/codex/gpt-5.6-luna/judge-v1/production-batch1
```

Rerunning with unchanged settings and cache resumes validated judgments. Proceed
to Stage B only after the base has completed and written `memory.json`, rather
than `memory.incomplete.json`. A fresh build can require multiple invocations;
do not interpret one successful pilot as full-graph completion.

**Stage B — Astra pattern synthesis.** Read the completed Luna base and use the
four configured question areas to compare evidence across records. This preserves
the base and adds the pattern layer. The selected run records 23 synthesis calls.

```bash
MEMORY_CACHE_DIR=artifacts/cache \
python run_abstraction.py \
  --memory artifacts/hosted/codex/gpt-5.6-luna/judge-v1/production-batch1/memory.json \
  --questions config/abstraction_questions.json \
  --provider codex --model gpt-6-astra --max-calls 40 \
  --output artifacts/abstraction-gpt-6-astra
```

The abstraction CLI sets medium reasoning. Its historical call allowance is not
retained in the selected artifact; `40` above is the script's default allowance,
not a claim that 40 calls were used. `--dry-run` inspects the plan without calls.

**Submission location.** The completed Stage B directory was subsequently renamed
from `artifacts/abstraction-gpt-6-astra/` to **`artifacts/memory/`**. The JSON files
were unchanged. The final memory therefore contains both the Luna-built base
and Astra-generated patterns; it is not an Astra-only extraction.

These are documentation-only shell blocks. Rebuilding requires the supplied raw
graph and provider authentication. For a new reproduction, use fresh output and
cache directories under `artifacts/rebuild/`, pass the new base path to Stage B,
and preserve the submitted snapshot. New model generations need not be identical.

## 2. Load the selected artifact

Install `requirements.txt` and select that environment as the notebook kernel.
Run from the repository root (or a subdirectory). The tables below are computed
from the saved JSON. Quotations are displayed verbatim and escaped as text.

In [1]:
from collections import Counter
import hashlib
import html
import json
from pathlib import Path
import sys
from IPython.display import display, HTML, Markdown

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'artifacts/memory/memory.json').is_file()
             and (p / 'org_memory').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Run from the repository containing artifacts/memory/memory.json.')
sys.path.insert(0, str(ROOT))
from org_memory import get_item, RecallIndex, validate_memory

MEMORY_PATH = ROOT / 'artifacts/memory/memory.json'
original_sha256 = hashlib.sha256(MEMORY_PATH.read_bytes()).hexdigest()
memory = json.loads(MEMORY_PATH.read_text())
base = memory['build_metadata']
abstraction = memory['abstraction_metadata']

def show_table(rows):
    rows = list(rows)
    if not rows:
        display(Markdown('_No rows._'))
        return
    columns = list(dict.fromkeys(k for row in rows for k in row))
    def text(value):
        if value is None:
            return 'unknown / unscored'
        if isinstance(value, (dict, list)):
            return json.dumps(value, ensure_ascii=False)
        return str(value)
    head = '<tr>' + ''.join('<th>' + html.escape(c) + '</th>' for c in columns) + '</tr>'
    body = ''.join('<tr>' + ''.join('<td>' + html.escape(text(row.get(c))) + '</td>'
                                  for c in columns) + '</tr>' for row in rows)
    display(HTML('<div class="task1-results"><table>' + head + body + '</table></div>'))

def show_evidence(evidence_ids):
    rows = []
    for eid in dict.fromkeys(evidence_ids):
        e = memory['evidence'][eid]
        rows.append({'evidence_id': eid, 'record': e['record_id'],
                     'provenance': e['provenance']['kind'], 'relation': e['relation'],
                     'original text': e['source_span'] or e['provenance'].get('rationale', '')})
    show_table(rows)

display(HTML('<style>.task1-results{overflow-x:auto;margin:12px 0}'
             '.task1-results table{border-collapse:collapse;font-size:13px}'
             '.task1-results th,.task1-results td{border:1px solid #ccc;padding:8px;'
             'text-align:left;vertical-align:top;white-space:pre-wrap;'
             'min-width:100px;max-width:520px;overflow-wrap:anywhere}'
             '.task1-results th{background:#edf2f7;color:#18232f}</style>'))
show_table([{'artifact': str(MEMORY_PATH.relative_to(ROOT)),
             'schema': memory['schema_version'], 'SHA-256': original_sha256}])

artifact,schema,SHA-256
artifacts/memory/memory.json,3.1,c15c0651545d69d1e973ec5a6ebc935286d27f5ac48b5fafabc0acb0f9c0953d


## 3. Validate completion and inspect provenance

Validation replays stored judgments and checks raw evidence integrity, citations,
base preservation and pattern construction. It does not call a provider or establish
semantic correctness. This checks the archive stored in the memory; an independent
comparison with the supplied raw graph is an additional, optional check below.

The base metadata still reports zero patterns because it describes Stage A.
The actual `patterns` collection and `abstraction_metadata` describe Stage B.

In [2]:
assert memory['schema_version'] == '3.1'
assert base['status'] == 'complete' and base['scope'] == 'full_graph'
assert abstraction['status'] == 'complete' and abstraction['execution_scope'] == 'production'
validation = validate_memory(memory, raise_on_error=True)
show_table([validation])
show_table([
    {'stage': 'Base episodes, facts and outcomes', 'CLI': 'run_task1.py',
     'model': base['hosted_model'], 'provider': base['hosted_provider'],
     'reasoning': base['hosted_reasoning_effort'],
     'batch size': base['config']['llm_batch_size'], 'status': base['status'],
     'processed': base['coverage']['reviewed_candidates'],
     'planned': base['coverage']['selected_candidates']},
    {'stage': 'Pattern synthesis', 'CLI': 'run_abstraction.py',
     'model': abstraction['settings']['model'], 'provider': abstraction['settings']['provider'],
     'reasoning': abstraction['settings']['reasoning_effort'],
     'batch size': 'evidence bundles per synthesis packet', 'status': abstraction['status'],
     'processed': abstraction['coverage']['processed_packets'],
     'planned': abstraction['coverage']['planned_packets']},
])
show_table([{'collection': k, 'count': len(memory[k])}
            for k in ('nodes', 'evidence', 'episodes', 'facts', 'outcome_assessments', 'patterns')])
show_table([{'evidence kind': k, 'count': v}
            for k, v in sorted(Counter(e['provenance']['kind'] for e in memory['evidence'].values()).items())])

passed,errors,items_checked,evidence_checked,limitation
True,[],1785,4051,Structural validation cannot establish semantic entailment or real-world truth.


stage,CLI,model,provider,reasoning,batch size,status,processed,planned
"Base episodes, facts and outcomes",run_task1.py,gpt-5.6-luna,codex,medium,1,complete,816,816
Pattern synthesis,run_abstraction.py,gpt-6-astra,codex,medium,evidence bundles per synthesis packet,complete,23,23


collection,count
nodes,3098
evidence,4051
episodes,653
facts,1086
outcome_assessments,791
patterns,46


evidence kind,count
extracted,3239
inferred,812


## 4. Episodes: preserve an incident and its evidence

The government-tenant example separates the reported absence of narrative updates,
timeout behavior, and intervention from unverified causal claims. Episode grouping
and retention were judged by Luna. Titles come from unverified graph labels;
approval of a group does not verify every attached passage or relation.
The archive below includes inferred context, explicitly labeled as such.

In [3]:
episode = next(e for e in memory['episodes'] if 'KEP-6969' in e['record_ids'])
show_table([{k: episode[k] for k in ('id', 'record_ids', 'title', 'title_origin',
                                    'retention', 'retention_reason', 'uncertainties')}])
show_table([{'claim': c['text'], 'status': c['status'], 'evidence_ids': c['evidence_ids']}
            for c in episode['claims']])
show_evidence(episode['evidence_ids'])

id,record_ids,title,title_origin,retention,retention_reason,uncertainties
episode_69c0485386f1acbe,"[""KEP-6969""]",increase processing timeout from 10 to 12 hours,unverified graph decision labels,retained,"Evidence coherently describes one customer outage, its cause, and timeout decision; significant customer impact makes it worth remembering.","[""Accepted grouping does not verify every excerpt, graph label or causal claim."", ""Fixed boundaries can conflate occurrences or omit related context; no automatic repair."", ""Compression reduces default recall prominence; all source excerpts remain available.""]"


claim,status,evidence_ids
"A government customer network in the Argonaut tenant (network_id: ca95298f-cb34-5119-b8f9-e447607dfab7) stopped generating narrative updates from approximately April 19, 2026",source_excerpt,"[""ev_1c1739cdda90321d""]"
"A government customer network in the Argonaut tenant (network_id: ca95298f-cb34-5119-b8f9-e447607dfab7) stopped generating narrative updates from approximately April 19, 2026. The issue was not caught by any internal alarm — it was discovered by the government customer's own team, then, when the issue continued, our alarms caught it but no action was taken. Root cause: narrative generation jobs for this network were repeatedly running beyond the processing time SLA threshold (10 hours at the time) and being killed by the application before completing.",source_excerpt,"[""ev_9b4317c218937e4f""]"
Government customer (Argonaut tenant) went 10+ days (Apr 19–29) without narrative updates for at least one network,source_excerpt,"[""ev_4386d80195ef6b5a""]"
Narrative generation jobs for networks with large data volumes (long-running networks) were being terminated by the application upon exceeding the processing SLA timeout (10 hours),source_excerpt,"[""ev_f3dba2eba43e2de5""]"
Root cause: narrative generation jobs for this network were repeatedly running beyond the processing time SLA threshold (10 hours at the time) and being killed by the application before completing,source_excerpt,"[""ev_5f2b2e2ded8842f5""]"
"The same job would then re-queue and be killed again, creating a silent retry loop with no output and no alert",source_excerpt,"[""ev_1b68761b66cc3cc3""]"
the processing timeout was increased from 10 hours to 12 hours,source_excerpt,"[""ev_ad26bffaa983949e""]"


evidence_id,record,provenance,relation,original text
ev_1b68761b66cc3cc3,KEP-6969,extracted,REFERENCES,"The same job would then re-queue and be killed again, creating a silent retry loop with no output and no alert"
ev_1c1739cdda90321d,KEP-6969,extracted,REFERENCES,"A government customer network in the Argonaut tenant (network_id: ca95298f-cb34-5119-b8f9-e447607dfab7) stopped generating narrative updates from approximately April 19, 2026"
ev_4386d80195ef6b5a,KEP-6969,extracted,PRODUCES,Government customer (Argonaut tenant) went 10+ days (Apr 19–29) without narrative updates for at least one network
ev_5f2b2e2ded8842f5,KEP-6969,extracted,REFERENCES,Root cause: narrative generation jobs for this network were repeatedly running beyond the processing time SLA threshold (10 hours at the time) and being killed by the application before completing
ev_9b4317c218937e4f,KEP-6969,extracted,CAUSES,"A government customer network in the Argonaut tenant (network_id: ca95298f-cb34-5119-b8f9-e447607dfab7) stopped generating narrative updates from approximately April 19, 2026. The issue was not caught by any internal alarm — it was discovered by the government customer's own team, then, when the issue continued, our alarms caught it but no action was taken. Root cause: narrative generation jobs for this network were repeatedly running beyond the processing time SLA threshold (10 hours at the time) and being killed by the application before completing."
ev_ad26bffaa983949e,KEP-6969,extracted,AFFECTS,the processing timeout was increased from 10 hours to 12 hours
ev_f3dba2eba43e2de5,KEP-6969,extracted,AFFECTS,Narrative generation jobs for networks with large data volumes (long-running networks) were being terminated by the application upon exceeding the processing SLA timeout (10 hours)


## 5. Facts and outcomes: distinguish reports from prescriptions

A retained historical statement reports what a source says happened. Requirements
and documented routines describe prescribed behavior; they do not prove execution.
These three selected facts show those distinctions with their source references.
Labels below are stored model judgments, not independently adjudicated truth.

In [4]:
show_table([{'fact kind': k, 'count': v}
            for k, v in sorted(Counter(f['kind'] for f in memory['facts']).items())])
fact_ids = ['fact_1d79bf4d32849728', 'fact_001b9d9caf7ebb77', 'fact_b539b2b4cbe85407']
selected_facts = [get_item(memory, fid) for fid in fact_ids]
show_table([{k: f[k] for k in ('id', 'kind', 'statement', 'record_ids', 'entity_ids', 'uncertainties')}
            for f in selected_facts])
show_evidence([eid for f in selected_facts for eid in f['evidence_ids']])

show_table([{'observed': observed, 'valence': valence, 'count': count}
            for (observed, valence), count in sorted(Counter(
                (o['observed'], o['valence']) for o in memory['outcome_assessments']).items())])
selected_outcomes = [
    next(o for o in memory['outcome_assessments'] if o['method'] == 'exact_administrative_guard'),
    next(o for o in memory['outcome_assessments'] if 'KEP-6969' in o['record_ids']),
]
show_table([{k: o[k] for k in ('id', 'record_ids', 'observed', 'valence', 'impact',
                              'signed_impact', 'reason', 'causal_attribution')}
            for o in selected_outcomes])
show_evidence([eid for o in selected_outcomes for eid in o['evidence_ids']])

fact kind,count
documented_routine,63
historical_statement,551
requirement,472


id,kind,statement,record_ids,entity_ids,uncertainties
fact_1d79bf4d32849728,historical_statement,Government customer (Argonaut tenant) went 10+ days (Apr 19–29) without narrative updates for at least one network,"[""KEP-6969""]","[""outcome_ab39ce9c""]","[""Source-supported historical knowledge; not independently verified current state."", ""Requirements and routines do not establish execution.""]"
fact_001b9d9caf7ebb77,requirement,Acceptance criteria Successfully integrate the BrightData API for account generation. Ensure that the new integration overcomes the limitations of the serp API. Validate that the output generated by BrightData API is accurate. No change in output structure of tools.,"[""KEP-6661""]","[""rule_0f1d5908""]","[""Source-supported historical knowledge; not independently verified current state."", ""Requirements and routines do not establish execution.""]"
fact_b539b2b4cbe85407,documented_routine,"This is a manual QA activity to execute the Postman collection twice daily (morning and evening) throughout the month. The purpose is to monitor API health, validate responses, and identify any failures early.","[""KT-292""]","[""resource_4169dfd2""]","[""Source-supported historical knowledge; not independently verified current state."", ""Requirements and routines do not establish execution.""]"


evidence_id,record,provenance,relation,original text
ev_4386d80195ef6b5a,KEP-6969,extracted,PRODUCES,Government customer (Argonaut tenant) went 10+ days (Apr 19–29) without narrative updates for at least one network
ev_e41f75e052e1effd,KEP-6661,extracted,REFERENCES,Acceptance criteria Successfully integrate the BrightData API for account generation. Ensure that the new integration overcomes the limitations of the serp API. Validate that the output generated by BrightData API is accurate. No change in output structure of tools.
ev_72bcaf2a50c998b3,KT-292,extracted,AFFECTS,"This is a manual QA activity to execute the Postman collection twice daily (morning and evening) throughout the month. The purpose is to monitor API health, validate responses, and identify any failures early."


observed,valence,count
False,unknown,639
True,beneficial,94
True,harmful,9
True,mixed,4
True,unknown,45


id,record_ids,observed,valence,impact,signed_impact,reason,causal_attribution
valuation_002d787eadb28da2,"[""KEP-2231""]",False,unknown,unknown,unknown / unscored,Administrative closure or missing excerpt cannot establish an observed result.,Unverified; a PRODUCES edge does not establish causality.
valuation_a7bad649411ca36e,"[""KEP-6969""]",True,harmful,customer-level,-3,The passage reports an actual 10+ day absence of narrative updates for a government customer network.,Unverified; a PRODUCES edge does not establish causality.


evidence_id,record,provenance,relation,original text
ev_28874ac431368aa9,KEP-2231,extracted,PRODUCES,status: Done
ev_4386d80195ef6b5a,KEP-6969,extracted,PRODUCES,Government customer (Argonaut tenant) went 10+ days (Apr 19–29) without narrative updates for at least one network


`status: Done` is administrative closure, not proof of successful recovery. An
observed harmful outcome above means the source reports harm. The signed impact
is an ordinal sign × scope weight (local=1, service=2, customer=3), not measured
business value. Mixed and unknown results remain unscored. A `PRODUCES` edge does
not establish causality between an intervention and an outcome.

## 6. Patterns: Astra compares the completed Luna base

The final 46 patterns consist of reported recurrences, hypothesized mechanisms
and prescribed routines. Inspect all three categories below. Each pattern has
exact support, links to episodes/facts, comparison insight, counterevidence and
uncertainty. A structural support-group count is a conservative proxy for
independent occurrences, not proof of independence or causal validity.

In [5]:
show_table([{'pattern kind': k, 'count': v}
            for k, v in sorted(Counter(p['kind'] for p in memory['patterns']).items())])
pattern_ids = ['pattern_780a6edcbfcaacb4', 'pattern_a1665b6ced1f35a6', 'pattern_f4d689e47e00fa12']
for pid in pattern_ids:
    pattern = get_item(memory, pid)
    display(Markdown('### ' + pattern['kind'].replace('_', ' ')))
    show_table([{k: pattern[k] for k in ('id', 'statement', 'comparison_insight',
                'differences_counterevidence', 'uncertainty', 'usefulness',
                'independent_support_count', 'supporting_episode_ids', 'supporting_fact_ids')}])
    display(Markdown('**Supporting source passages**'))
    show_evidence(pattern['evidence_ids'])
    display(Markdown('**Counterevidence / qualifying source passages**'))
    show_evidence(pattern['counterevidence_evidence_ids'])

pattern kind,count
hypothesized_mechanism,8
prescribed_routine,12
reported_recurrence,26


### reported recurrence

id,statement,comparison_insight,differences_counterevidence,uncertainty,usefulness,independent_support_count,supporting_episode_ids,supporting_fact_ids
pattern_780a6edcbfcaacb4,"A stalled local state can reflect missing or unusable completion signals, not necessarily unfinished computation.",Bright Data reportedly finished processing without reliable local completion transitions; comments-stance tasks lacked callbacks after execution; a callback type mismatch prevented completion marking elsewhere. Video-Summary also omitted failure callbacks. Together these distinguish execution reliability from reliable reporting of terminal outcomes.,"Not every waiting task had executed: the Dagster incident reported successful S3 upload but no downstream start. Its pending jobs subsequently completed with successful backfill and no data loss, unlike the unresolved callback cases.","These are source-reported occurrences, not independently verified diagnoses. Similar symptoms do not establish a shared defect, and the selected records cannot establish prevalence.","Helps a lead separate execution failures from state-reconciliation failures before choosing replay, callback repair, or reconciliation; a generic retry could repeat already-performed work.",4,"[""episode_23141a4207bb17b0"", ""episode_4c6635c9e9e6b0e4"", ""episode_ad68a997dd1184a5"", ""episode_e1df026e607f0fc0""]","[""fact_2455fe9d37ed6be2"", ""fact_2d4b3e93c558bf7d"", ""fact_e4f46a4c7efe0556""]"


**Supporting source passages**

evidence_id,record,provenance,relation,original text
ev_2387deb7a51273b0,KEP-5668,extracted,CAUSES,"Due to a bug, batch statuses are not reliably transitioning from sent → completed after Bright Data finishes processing."
ev_615c89da13c2fed2,KEP-5588,extracted,REFERENCES,Some comments stance tasks do not receive a service callback even after task execution.
ev_90b8b0d6052b3676,KEP-2175,extracted,REFERENCES,Callbacks missing for failure cases Callbacks are not being triggered when Video-Summary tasks fail.
ev_cccf85ea01d19cfa,KEP-5641,extracted,REFERENCES,"Key: Root cause: callback_status is sent as a string instead of an integer, so the background service does not mark tasks as complete, leaving the task group stuck in an incomplete state."


...

id,statement,comparison_insight,differences_counterevidence,uncertainty,usefulness,independent_support_count,supporting_episode_ids,supporting_fact_ids
pattern_f4d689e47e00fa12,"Two monitoring records prescribe a common twice-daily, dashboard-based decline check with Slack escalation, while zero-value checks add a distinct detection condition.",The matching cadence and sustained-decline threshold suggest a reusable detection routine. Trend degradation and zero-value conditions are distinct signals rather than a single generic failure threshold.,"One record explicitly assigns high priority to a zero-value alert; the other supplied excerpts specify decline monitoring and Slack notification without that priority. Separately, display requirements recognize legitimate zero activity, so a zero alert alone is not proof of pipeline failure. No execution counterexample is supplied.","These prescriptions do not demonstrate checks performed, alerts delivered, or incidents resolved. Shared wording may reflect reused documentation; selected excerpts cannot establish coverage or prevalence.","Provides a basis for standardizing alert criteria and auditing actual monitoring execution, while keeping notification distinct from remediation.",2,"[""episode_3c44e67d3a8c1c5e"", ""episode_fc00e2d6d08f50b8""]","[""fact_069ce50d68ac91a5"", ""fact_bdeefecdf5b84986""]"


**Supporting source passages**

evidence_id,record,provenance,relation,original text
ev_12bedb14a8af7e0c,KEP-6559,extracted,REFERENCES,"If any value = 0 , alert in #kepler-data-validation (Slack) with High Priority"
ev_4d8b66397755aa2d,KEP-6736,extracted,REFERENCES,alert in #kepler-data-validation (Slack)
ev_9f543bc4c976ff23,KEP-6559,extracted,REFERENCES,"This activity should be performed twice a day to ensure continuous monitoring of data consistency, completeness, and anomaly detection"
ev_a5524b6316553c2f,KEP-6559,extracted,REFERENCES,Condition to Monitor: ⚠️ Risky - Platforms with 2+ consecutive days of decline (day-over-day drop > 5%)
ev_dafbb62dc31a2722,KEP-6736,extracted,REFERENCES,Platforms with 2+ consecutive days of decline (day-over-day drop > 5%)
ev_fd56f8eec9a2178f,KEP-6736,extracted,REFERENCES,This activity should be performed twice a day to ensure continuous monitoring


**Counterevidence / qualifying source passages**

evidence_id,record,provenance,relation,original text
ev_12bedb14a8af7e0c,KEP-6559,extracted,REFERENCES,"If any value = 0 , alert in #kepler-data-validation (Slack) with High Priority"
ev_49dfbaf7a812db8a,KEP-7092,extracted,REFERENCES,"Metrics (post counts, comment counts, like counts, influence scores, view counts) must never display zero as a result of a pipeline failure. Only as a genuine reflection of zero activity"
ev_d9c5cd5791cb4cbe,KEP-6559,extracted,REFERENCES,"If any value = 0 , alert in #kepler-data-validation (Slack)"


### What the pattern stage covered and rejected

Synthesis is question-focused: four investigation areas route selected evidence
bundles into packets. Completion means all planned packets were processed; it
does not mean exhaustive discovery across the organization. Primary and context
sets can overlap, so their sizes should not be summed as disjoint coverage.
The rejected proposal below illustrates that a plausible comparison can fail the
admission contract; rejection is not proof that its underlying events are false.

In [6]:
coverage = abstraction['coverage']
show_table([{'question': q['question'], 'routing cues': q['cues']}
            for q in abstraction['questions']])
show_table([{
    'prepared bundles': coverage['total_bundles'],
    'primary bundles': len(coverage['selected_bundle_ids']),
    'context bundles': len(coverage['context_bundle_ids']),
    'omitted bundles': len(coverage['omitted_bundle_ids']),
    'processed packets': coverage['processed_packets'],
    'retained patterns': len(memory['patterns']),
    'rejected proposals': len(abstraction['rejected_proposals']),
}])
rejection = abstraction['rejected_proposals'][0]
show_table([{'packet': rejection['packet_id'],
             'proposed statement': rejection['proposal']['statement'],
             'proposed kind': rejection['proposal']['kind'], 'reason': rejection['reason']}])
display(Markdown('The full [inspection examples](artifacts/memory/inspection_examples.json), '
                 '[rejections](artifacts/memory/rejected_proposals.json) and '
                 '[coverage report](artifacts/memory/coverage.json) retain the other cases.'))

question,routing cues
"What do the records reveal about reliability, detection, response, and their differences across situations?","[""retry"", ""failure"", ""failed"", ""crash"", ""crashed"", ""timeout"", ""alert"", ""monitoring"", ""incident"", ""recovery"", ""recovered"", ""success"", ""pipeline"", ""queue"", ""stuck"", ""restart""]"
"How do storage and data consistency behaviors compare across the records, including successful cases?","[""DocDB"", ""Elasticsearch"", ""storage"", ""consistency"", ""inconsistent"", ""mismatch"", ""discrepancy"", ""cursor"", ""sync"", ""partial"", ""duplicate""]"
"What operational routines are described, and what evidence distinguishes prescriptions from reported execution?","[""routine"", ""health"", ""check"", ""checks"", ""runbook"", ""daily"", ""weekly"", ""operational"", ""maintenance"", ""validation"", ""deployment""]"
"How does narrative and pruning behavior vary across records, actions, and outcomes?","[""narrative"", ""pruning"", ""prune"", ""hidden"", ""visibility"", ""reappear"", ""refresh""]"


prepared bundles,primary bundles,context bundles,omitted bundles,processed packets,retained patterns,rejected proposals
697,293,42,376,23,46,22


packet,proposed statement,proposed kind,reason
packet_1eeaad0b85f114d5,"Queue and worker separation recurs as a reported engineering response, but the records demonstrate configuration changes more clearly than reliability improvements.",reported_recurrence,Reported recurrence requires reported occurrences


The full [inspection examples](artifacts/memory/inspection_examples.json), [rejections](artifacts/memory/rejected_proposals.json) and [coverage report](artifacts/memory/coverage.json) retain the other cases.

## 7. Retention: reduce prominence without deleting evidence

Of the 653 accepted episodes, 547 are retained for default recall and 106 are
compressed. Here, compression means exclusion from default recall, while all
source excerpts remain in the archive. It does not shorten the stored excerpts
or implement age-based decay. Useful facts can survive a rejected episode.
This distinction keeps low-value administrative history inspectable without
treating it as a useful default answer.

In [7]:
retriever = RecallIndex(memory)
inclusive_retriever = RecallIndex(memory, include_compressed=True)
show_table([{'episode retention': k, 'count': v}
            for k, v in sorted(Counter(e['retention'] for e in memory['episodes']).items())])
compressed = next(e for e in memory['episodes'] if e['retention'] == 'compressed')
assert compressed['id'] not in retriever.by_id
assert compressed['id'] in inclusive_retriever.by_id
assert all(eid in memory['evidence'] for eid in compressed['evidence_ids'])
show_table([{'episode': compressed['id'], 'records': compressed['record_ids'],
             'retention reason': compressed['retention_reason'],
             'default recall eligible': compressed['id'] in retriever.by_id,
             'include_compressed eligible': compressed['id'] in inclusive_retriever.by_id,
             'preserved source passages': len(compressed['evidence_ids'])}])
show_table([{'archived candidate groups': len(memory['archived_candidates']),
             'archived fact proposals': len(memory['archived_facts']),
             'all preserved evidence records': len(memory['evidence'])}])
show_evidence(compressed['evidence_ids'])

episode retention,count
compressed,106
retained,547


episode,records,retention reason,default recall eligible,include_compressed eligible,preserved source passages
episode_005cd179e617c63a,"[""KEP-3480"", ""KEP-6883""]","The passages support a connected validation action, but provide little beyond a procedure and administrative completion status.",False,True,2


archived candidate groups,archived fact proposals,all preserved evidence records
163,1825,4051


evidence_id,record,provenance,relation,original text
ev_738b1bb17b331acc,KEP-6883,extracted,AFFECTS,"After restarting the Elasticsearch (ES) Nodes, perform end-to-end validation to ensure all functionalities and dependent services are operating correctly"
ev_e576abb4d5182f41,KEP-3480,extracted,PRODUCES,status: Done


## 8. Recall from partial cues

This portable demonstration uses **BM25 lexical recall plus bounded memory
associations**, without the raw graph's node embeddings and without hosted
embeddings. Scores are ranking heuristics, not calibrated confidence or evidence
that a claim is true. Associations link patterns with supporting episodes/facts
and scoped entities. The development cues illustrate behavior, not retrieval
accuracy on held-out labels. Their results can differ from runs with node vectors.

In [8]:
cues = [
    'Jobs keep retrying but users see no output',
    'Pruning rules deleted but narratives remain hidden',
    'Routine API health checks',
    'Data differs between stores',
    'DocDB Elasticsearch incomplete storage and drift',
]
recalls = {cue: retriever.recall(cue, limit=4) for cue in cues}
for cue, result in recalls.items():
    display(Markdown('### ' + cue))
    show_table([{'rank': rank, 'id': r['item_id'], 'type': r['type'], 'title': r['title'],
                 'records': r['record_ids'], 'score': r['score'], 'retrieval reason': r['why']}
                for rank, r in enumerate(result['results'], 1)])
first = recalls[cues[0]]
show_table([{'query': cues[0], 'base retrieval mode': first['trace']['mode'],
             'association hop limit': first['trace']['max_hops'],
             'score interpretation': first['trace']['score_note']}])
best = get_item(memory, first['results'][0]['item_id'])
display(Markdown('**Open the top item and follow its exact evidence**'))
show_table([{'id': best['id'], 'type': best['type'], 'statement': best.get('statement', best['summary']),
             'records': best['record_ids'], 'uncertainties': best.get('uncertainties', [])}])
show_evidence(best['evidence_ids'])

### Jobs keep retrying but users see no output

rank,id,type,title,records,score,retrieval reason
1,fact_69328914c3d11d7c,fact,Cascading failures in downstream ML jobs because the stats service cannot keep up.,"[""KEP-6189""]",0.01856682,lexical; memory_association; supports this cross-record pattern → fact supplies evidence for this pattern
2,fact_045ac6dd603c43e4,fact,"Task retries: Jobs retrying up to 3x in Dexter, inflating GPU usage.","[""Community_1981""]",0.01842755,lexical; memory_association; supports this cross-record pattern → fact supplies evidence for this pattern
3,fact_8eab714654eebf3d,fact,"BrightData batching service may be temporarily unavailable (pod restart, rollout, scaling). No retry mechanism exists for connection refused / network-level failures . Causes: Unnecessary task failures Missed BrightData job creation Inconsistent ingestion behavior","[""KEP-5076""]",0.01737879,lexical; memory_association; supports this cross-record pattern → fact supplies evidence for this pattern
4,fact_2ea95543a318e7be,fact,"When the network builder fails to generate account suggestions (error: ""Failed to generate accounts""), there is no retry mechanism available to the user. The flow becomes a dead end. No retry button","[""KEP-6518""]",0.01713636,lexical; memory_association; supports this cross-record pattern → fact supplies evidence for this pattern


### Pruning rules deleted but narratives remain hidden

rank,id,type,title,records,score,retrieval reason
1,episode_421751971a082de3,episode,Investigate and resolve bug where deleted pruning rule does not restore narrative to main narrative tab,"[""Community_1210"", ""KEP-4019"", ""KEP-5342""]",0.01827331,lexical; memory_association; supports this cross-record pattern → episode supplies evidence for this pattern
2,fact_de93306eb83782b2,fact,Even after deleting the pruning rule: The Narrative does not reappear in the main Narrative tab. The Narrative remains hidden. Suppression state appears to persist incorrectly.,"[""KEP-5342""]",0.01813839,lexical; memory_association; supports this cross-episode pattern → fact supplies evidence for this pattern
3,pattern_8af495de8d8dd82f,pattern,"Rule deletion has two separately reported failure modes: narratives remain hidden, and the UI does not automatically refresh.","[""KEP-5342"", ""KEP-5389"", ""KEP-5956""]",0.01809524,lexical; memory_association; supports this cross-episode pattern
4,episode_0e9c369dbf01c1a7,episode,Fix UI synchronization issue where deleted pruning rules do not refresh immediately,"[""KEP-4210"", ""KEP-5389"", ""KEP-5480"", ""KEP-5737"", ""KEP-6216""]",0.01773593,lexical; memory_association; supports this cross-episode pattern → episode supplies evidence for this pattern


### Routine API health checks

...

rank,id,type,title,records,score,retrieval reason
1,pattern_04c3d74eaa9b6cbf,pattern,Storage failures span both incomplete persistence and divergence between stores; these are distinct consistency checks.,"[""Community_141"", ""KEP-6611"", ""KEP-6664""]",0.01888154,lexical; memory_association; supports this cross-episode pattern
2,episode_0099b4eefc7521d8,episode,Fix DocDB and Elasticsearch drift from partial de-duplication updates,"[""KEP-6230"", ""KEP-6664""]",0.01856682,lexical; memory_association; supports this cross-episode pattern → episode supplies evidence for this pattern
3,episode_2ea1ec7fc40462d2,episode,Fix inconsistent DocDB storage due to incorrect cursor handling,"[""KEP-6230"", ""KEP-6611""]",0.01842755,lexical; memory_association; supports this cross-episode pattern → episode supplies evidence for this pattern
4,fact_3c1d77d75cf616dc,fact,DocDB/ES drift from partial de-duplication updates,"[""KEP-6664""]",0.01821673,lexical; memory_association; supports this cross-episode pattern → fact supplies evidence for this pattern


query,base retrieval mode,association hop limit,score interpretation
Jobs keep retrying but users see no output,BM25 only,2,"RRF k=60; support multipliers 1/.85/.65; association decay .6 per hop. Heuristics, not probabilities."


**Open the top item and follow its exact evidence**

id,type,statement,records,uncertainties
fact_69328914c3d11d7c,fact,Cascading failures in downstream ML jobs because the stats service cannot keep up.,"[""KEP-6189""]","[""Source-supported historical knowledge; not independently verified current state."", ""Requirements and routines do not establish execution.""]"


evidence_id,record,provenance,relation,original text
ev_e29d92b08771767b,KEP-6189,extracted,REFERENCES,Cascading failures in downstream ML jobs because the stats service cannot keep up.


## 9. Construction cost and a known surviving error

Report construction cost separately from notebook inspection or per-question
answering cost. The base summary below is cumulative across resumes; the last
invocation's smaller usage summary would undercount the completed build. Logged
provider seconds are accumulated request latency, not an end-to-end wall-clock
measurement. Reported usage can omit failed calls, and subscription dollar cost
is unknown. Cached input tokens are a subset of input tokens, not an extra total.

In [9]:
cost_rows = []
for stage, usage in [('Luna base: cumulative', base['cumulative_usage_summary']),
                     ('Astra patterns', abstraction['usage_summary'])]:
    tokens = usage['reported_new_call_tokens']
    cost_rows.append({'stage': stage, 'measured requests': usage['measured_requests'],
                      'input tokens': tokens.get('input_tokens'),
                      'cached input tokens': tokens.get('cached_input_tokens'),
                      'output tokens': tokens.get('output_tokens'),
                      'summed provider seconds': usage['elapsed_provider_seconds'],
                      'failed requests': usage['failed_requests'],
                      'estimated dollars': usage['estimated_this_run_cost_usd']})
show_table(cost_rows)

known_error = get_item(memory, 'fact_59af81ef6bfea7e3')
show_table([{'fact': known_error['id'], 'statement': known_error['statement'],
             'assigned entities': [{'id': eid, 'graph label': memory['nodes'][eid]['label']}
                                   for eid in known_error['entity_ids']],
             'records': known_error['record_ids']}])
show_evidence(known_error['evidence_ids'])

stage,measured requests,input tokens,cached input tokens,output tokens,summed provider seconds,failed requests,estimated dollars
Luna base: cumulative,838,3795381,1872384,488146,8713.1158,3,unknown / unscored
Astra patterns,23,300245,2560,26978,744.608,0,unknown / unscored


fact,statement,assigned entities,records
fact_59af81ef6bfea7e3,"Today, comment download completion is tracked only as a boolean flag at the post level (comments_downloaded: bool) in both: CommonPostMetadata (ElasticSearch) RedditPostMetadata (Mongo)","[{""id"": ""resource_08ce433d"", ""graph label"": ""Twitter comment download status tracking""}]","[""Community_534""]"


evidence_id,record,provenance,relation,original text
ev_9af8da11382aa82a,Community_534,extracted,AFFECTS,"Today, comment download completion is tracked only as a boolean flag at the post level (comments_downloaded: bool) in both: CommonPostMetadata (ElasticSearch) RedditPostMetadata (Mongo)"


The last example is a known subject-scoping failure: the passage refers to
`RedditPostMetadata`, but the stored fact is assigned to a Twitter resource.
The quotation can be exact while the entity assignment is unsupported. The
notebook preserves and exposes this error instead of silently repairing the
frozen artifact. A passing structural check is therefore not an accuracy score.

Other limits remain: fixed groups can conflate events; rejecting a whole excerpt
can lose useful subclaims; prescribed routines do not prove execution; retained
patterns have low heuristic confidence and may overstate independence or causality.
This inspection does not establish a downstream advantage over text RAG or graph
retrieval. That comparison belongs in Task 2. Generate its `task2_evaluation.ipynb`
source with `python scripts/create_evaluation_notebook.py`.

## 10. Optional source comparison and reproduction

To compare the memory archive with the original graph, place the supplied
`KEP_2026.json` at the repository root and run this **validation-only** CLI:

```bash
python run_task2.py --graph KEP_2026.json \
  --memory artifacts/memory/memory.json --validate-only
```

That command makes no model calls. For recall with the supplied node vectors,
pass `graph['id2embeddings']` as the second argument to `RecallIndex`; label that
mode explicitly. The original embedding encoder is unknown, so those vectors
do not enable direct embedding of new question text into a known space.

Generate or refresh this results notebook without regenerating memory:

```bash
python scripts/create_task1_results_notebook.py
python scripts/execute_notebook.py --notebook task1_results.ipynb --in-process
```

The generator clears notebook outputs. The executor fills them from the saved
artifact. `task1_hosted.ipynb` remains an optional live construction pilot; its
execution is not required to inspect these completed results.

In [10]:
assert hashlib.sha256(MEMORY_PATH.read_bytes()).hexdigest() == original_sha256
show_table([{'memory unchanged during inspection': True,
             'structural validation passed': validation['passed'],
             'base completion': base['status'], 'pattern completion': abstraction['status'],
             'model calls in this notebook': 0}])

memory unchanged during inspection,structural validation passed,base completion,pattern completion,model calls in this notebook
True,True,complete,complete,0
